# Baseline Soay Sheep IBM

This notebook reproduces the ungulate life cycle, retains the final completed simulation year, saves a canonical random sample of 3,000 sheep, and saves a 1,000-row subset for downstream GP work.

In [1]:
import numpy as np
import pandas as pd
from ipm_utils import calculate_ipm_metrics, mk_K_glm, reproduction_fn_glm, simulate_sheep_ibm

In [2]:
m_par_true = {"surv_int": -9.65, "surv_slope": 3.77, "growth_int": 1.41, "growth_slope": 0.557, "growth_noise": 0.0799, "repr_int": -7.23, "repr_slope": 2.60, "recr_int": 1.93, "rcsz_int": 0.362, "rcsz_slope": 0.709, "rcsz_noise": 0.159, "female_prob": 0.5}

In [3]:
rng = np.random.default_rng(270875)
result = simulate_sheep_ibm(m_par_true, n_years=400, init_pop_size=500, rng=rng, max_population=5000, retain="final")
final_year_data = result["data"]
if len(final_year_data) < 3000:
    raise ValueError(f"The last completed year contains only {len(final_year_data)} sheep; 3,000 are required")
rows = rng.choice(len(final_year_data), 3000, replace=False)
baseline_3000 = final_year_data.iloc[rows].sort_values("z").reset_index(drop=True)
baseline_1000 = baseline_3000.sample(1000, replace=False, random_state=53241986).sort_values("z").reset_index(drop=True)
baseline_3000.to_csv("baseline_ibm_data_3000.csv", index=False)
baseline_1000.to_csv("baseline_ibm_data_1000.csv", index=False)
print(f"Last Completed Year: {result['last_year']}")
print(f"Last-Year Population Before Sampling: {len(final_year_data)}")
print(f"Next Population Size: {result['next_population_size']}")

Last Completed Year: 161
Last-Year Population Before Sampling: 4914
Next Population Size: 5015


In [4]:
baseline_3000.describe().round(3)

,z,Surv,z1,Repr,Sex,Recr,Rcsz,yr
count,3000.000,3000.000,2358.000,2358.000,1558.000,790.000,687.000,3000.0
mean,2.978,0.786,3.106,0.661,0.507,0.870,2.548,161.0
std,0.289,0.410,0.152,0.474,0.500,0.337,0.200,0.0
min,1.778,0.000,2.449,0.000,0.000,0.000,1.895,161.0
25%,2.817,1.000,3.028,0.000,0.000,1.000,2.420,161.0
50%,3.076,1.000,3.129,1.000,1.000,1.000,2.559,161.0
75%,3.183,1.000,3.210,1.000,1.000,1.000,2.686,161.0
max,3.489,1.000,3.558,1.000,1.000,1.000,3.215,161.0


In [5]:
L, U, n_mesh = 1.6, 3.7, 250
true_ipm = mk_K_glm(n_mesh, m_par_true, L, U, correction=True)
mesh = true_ipm["mesh_points"]
true_metrics = calculate_ipm_metrics(true_ipm["K"], reproduction_fn_glm(mesh, m_par_true))
np.savez_compressed("true_baseline_metrics.npz", **true_metrics)
print(f"True Lambda: {true_metrics['lambda']:.6f}")

True Lambda: 1.020261
